# 🚦 Vehicle Counting & Classification — Smart City Diskominfo Banda Aceh
### YOLOv8 + ByteTrack + Supervision Line Zone Counting
**(Versi Perbaikan — sudah teruji jalan di Kaggle Notebook dengan metode `curl`)**

Notebook ini melakukan:
1. Instalasi environment (`ultralytics`, `supervision`, `roboflow`)
2. Pengunduhan dataset dari Roboflow — **metode `curl` sebagai jalur utama** (sudah terbukti berhasil), SDK `roboflow` sebagai alternatif opsional
3. Fine-tuning `yolov8n.pt` selama 50 epoch
4. Tracking kendaraan dengan **ByteTrack**
5. Penghitungan arus kendaraan (motor, mobil, truk, bus) dengan **Line Zone Counting** (Supervision)
6. Output akhir: `best.pt` + video hasil deteksi teranotasi (tracker ID + counting)

> ⚠️ **Catatan keamanan:** API key Roboflow di sel opsional bersifat rahasia. Jangan commit
> notebook ini ke repository publik selama sel tersebut masih berisi API key asli.
> Rotasi/ganti API key ini di dashboard Roboflow, lalu pindahkan ke **Kaggle Secrets**
> (Add-ons > Secrets) atau **Colab Secrets** (🔑 di sidebar) sebelum notebook dibagikan ke orang lain.

> 📌 **Catatan platform:** Jika dijalankan di **Kaggle**, pastikan **Internet** diaktifkan
> (klik ⚙️ Settings di sidebar kanan → toggle Internet = ON), dan aktifkan **GPU accelerator**
> (Settings → Accelerator → GPU T4 x2 atau P100) sebelum menjalankan sel training.


## 1. Setup Environment

In [1]:
# Cek GPU yang tersedia
# - Google Colab: Runtime > Change runtime type > GPU
# - Kaggle: Settings (sidebar kanan) > Accelerator > GPU
!nvidia-smi


Mon Jul 20 15:05:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install ultralytics supervision roboflow -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 7.6 MB/s eta 0:00:00


In [3]:
import os
import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO

print("Ultralytics & Supervision siap digunakan.")
print("Supervision version:", sv.__version__)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics & Supervision siap digunakan.
Supervision version: 0.29.1


## 2. Pengunduhan Dataset

**Jalur utama (Opsi B — `curl`)** sudah terbukti berhasil pada percobaan sebelumnya, jadi
dijadikan default di notebook ini. Opsi A (SDK Roboflow) disediakan sebagai alternatif opsional
di bagian bawah — jalankan hanya jika kamu ingin mengganti sumber dataset.


### ✅ Opsi B — `curl` + link ekspor Roboflow (Default / Terbukti Berhasil)

In [4]:
%%bash
set -e
curl -L "https://app.roboflow.com/ds/DpgTe77JoD?key=v6NVJeCDC9" > roboflow.zip
rm -rf dataset_curl
mkdir -p dataset_curl
unzip -o -q roboflow.zip -d dataset_curl
rm -f roboflow.zip
ls -la dataset_curl

total 32
drwxr-xr-x 5 root root 4096 Jul 20 15:06 .
drwxr-xr-x 3 root root 4096 Jul 20 15:06 ..
-rw-rw-r-- 1 root root  343 Jul 20 12:26 data.yaml
-rw-rw-r-- 1 root root  197 Jul 20 12:26 README.dataset.txt
-rw-rw-r-- 1 root root  880 Jul 20 12:26 README.roboflow.txt
drwxr-xr-x 4 root root 4096 Jul 20 15:06 test
drwxr-xr-x 4 root root 4096 Jul 20 15:06 train
drwxr-xr-x 4 root root 4096 Jul 20 15:06 valid


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   927  100   927    0     0   3193      0 --:--:-- --:--:-- --:--:--  3185
100  546M  100  546M    0     0   172M      0  0:00:03  0:00:03 --:--:--  212M


In [5]:
# PENTING: baris ini yang sebelumnya ter-lewat (comment) sehingga menyebabkan
# NameError: name 'DATASET_LOCATION' is not defined. Sekarang sudah aktif secara default.
DATASET_LOCATION = "dataset_curl"

print("Lokasi dataset yang dipakai:", DATASET_LOCATION)
!ls "{DATASET_LOCATION}"


Lokasi dataset yang dipakai: dataset_curl
data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


### (Opsional) Opsi A — Roboflow SDK (Python)

Jalankan sel ini **hanya jika** kamu ingin mengambil dataset lewat SDK alih-alih `curl`.
Sel ini bersifat *self-contained* (mendefinisikan `rf` sendiri) dan aman dijalankan sendirian
tanpa bergantung ke sel lain — sebelumnya error `NameError: name 'rf' is not defined' terjadi
karena sel diagnostik dipanggil sebelum `rf` dibuat; versi di bawah ini sudah memperbaikinya.

Jika sel ini berhasil dan kamu ingin memakainya, timpa `DATASET_LOCATION` di sel sebelumnya
dengan `dataset.location`.


In [6]:
from roboflow import Roboflow

# Sebaiknya ganti baris di bawah dengan Kaggle/Colab Secrets, contoh (Colab):
# from google.colab import userdata
# ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
ROBOFLOW_API_KEY = "sHqKubuxlapsZPfzDXCj"  # TODO: rotasi & pindahkan ke Secrets sebelum sharing notebook

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
print("Roboflow API key valid, koneksi berhasil.")

# --- Diagnostik: cek workspace, project (slug), dan versi yang benar-benar ada ---
# Untuk memastikan slug workspace/project yang benar, buka roboflow.com > project Anda >
# tab Versions > Download Dataset > pilih format YOLOv8 > "show download code", lalu
# sesuaikan nama workspace/project di bawah ini dengan yang ditampilkan.
try:
    ws = rf.workspace("firah-maulida")
    print("Workspace ditemukan:", ws)
    proj = ws.project("traffic-surveillance-system")
    print("Project ditemukan:", proj.name)
    available_versions = [v.version for v in proj.versions()]
    print("Versi yang tersedia:", available_versions)

    dataset = proj.version(1).download("yolov8")
    print("Dataset diunduh ke:", dataset.location)
    # Jika ingin memakai hasil Opsi A ini, jalankan baris berikut secara manual:
    # DATASET_LOCATION = dataset.location
except Exception as e:
    print("Gagal mengakses project via SDK:", repr(e))
    print("Tetap gunakan DATASET_LOCATION = 'dataset_curl' dari Opsi B di atas.")


Roboflow API key valid, koneksi berhasil.
loading Roboflow workspace...
Workspace ditemukan: {
  "name": "New Workspace",
  "url": "firah-maulida",
  "projects": [
    "firah-maulida/helmet-detection-j0oa1-pp9rj",
    "firah-maulida/traffic-density-izacs-2lgo3",
    "firah-maulida/traffic-density-jfihi-a3jsc",
    "firah-maulida/traffic-surveillance-system-wviaw",
    "firah-maulida/traffic-surveillance-system-ymhth",
    "firah-maulida/vehicle-detection-ph9yu-7ndl3",
    "firah-maulida/vehicle-detection-vkwyt-cmq3v"
  ]
}
loading Roboflow project...
Project ditemukan: Traffic surveillance system
Versi yang tersedia: ['8', '7', '2', '1']



Gagal mengakses project via SDK: BadZipFile('File is not a zip file')
Tetap gunakan DATASET_LOCATION = 'dataset_curl' dari Opsi B di atas.


## 3. Verifikasi `data.yaml`

Roboflow otomatis membuat `data.yaml` berisi path train/val/test dan daftar kelas.
Cek isinya sebelum training — pastikan nama kelas sesuai dengan kategori kendaraan yang
ingin dihitung (mis. `motor`, `mobil`, `truk`, `bus`, atau versi bahasa Inggrisnya
`motorcycle`, `car`, `truck`, `bus`, tergantung anotasi dataset aslinya).


In [7]:
import yaml

data_yaml_path = os.path.join(DATASET_LOCATION, "data.yaml")
with open(data_yaml_path, "r") as f:
    data_yaml = yaml.safe_load(f)

print(yaml.dump(data_yaml, allow_unicode=True))
CLASS_NAMES = data_yaml.get("names", [])
print("Kelas terdeteksi:", CLASS_NAMES)

names:
- bus
- car
- motorbike
- truck
nc: 4
roboflow:
  license: Public Domain
  project: traffic-surveillance-system-wviaw
  url: https://universe.roboflow.com/firah-maulida/traffic-surveillance-system-wviaw/dataset/dataset
  version: dataset
  workspace: firah-maulida
test: ../test/images
train: ../train/images
val: ../valid/images

Kelas terdeteksi: ['bus', 'car', 'motorbike', 'truck']


In [8]:
# (Opsional) Perbaiki path train/val/test di data.yaml jika Roboflow menuliskannya
# sebagai path relatif yang salah terhadap lokasi notebook. Jalankan sel ini kalau training
# di Bagian 4 gagal menemukan gambar dengan error semacam "no labels found" / path invalid.

abs_dataset_location = os.path.abspath(DATASET_LOCATION)

for split in ["train", "val", "test"]:
    if split in data_yaml:
        rel_path = data_yaml[split].replace("../", "")
        data_yaml[split] = os.path.join(abs_dataset_location, rel_path.split("/")[-2], rel_path.split("/")[-1]) \
            if "/" in rel_path else os.path.join(abs_dataset_location, rel_path)

with open(data_yaml_path, "w") as f:
    yaml.dump(data_yaml, f, allow_unicode=True)

print("data.yaml (path diperbaiki menjadi absolut):")
print(yaml.dump(data_yaml, allow_unicode=True))


data.yaml (path diperbaiki menjadi absolut):
names:
- bus
- car
- motorbike
- truck
nc: 4
roboflow:
  license: Public Domain
  project: traffic-surveillance-system-wviaw
  url: https://universe.roboflow.com/firah-maulida/traffic-surveillance-system-wviaw/dataset/dataset
  version: dataset
  workspace: firah-maulida
test: /kaggle/working/dataset_curl/test/images
train: /kaggle/working/dataset_curl/train/images
val: /kaggle/working/dataset_curl/valid/images



## 4. Fine-Tuning `yolov8n.pt` (50 Epoch)

Melakukan transfer learning dari bobot pretrained `yolov8n.pt` menggunakan dataset kendaraan.


In [9]:
model = YOLO("yolov8n.pt")

train_results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,          # early stopping jika tidak ada peningkatan
    project="vehicle_counting_runs",
    name="yolov8n_vehicle",
    exist_ok=True,
    plots=True,
)


Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_curl/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_vehicle, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

In [10]:
import glob

# PERBAIKAN: sebelumnya path ditulis manual "vehicle_counting_runs/..." dan gagal
# (AssertionError: best.pt tidak ditemukan), karena Ultralytics ternyata menyimpan hasil
# di bawah prefix tambahan "runs/detect/..." tergantung versi. Sekarang path diambil
# langsung dari model.trainer.save_dir (selalu akurat), dengan glob sebagai jaring pengaman.
save_dir = str(model.trainer.save_dir)
BEST_WEIGHTS_PATH = os.path.join(save_dir, "weights", "best.pt")

if not os.path.exists(BEST_WEIGHTS_PATH):
    candidates = glob.glob("/kaggle/working/**/yolov8n_vehicle/weights/best.pt", recursive=True)
    if not candidates:
        candidates = glob.glob("**/yolov8n_vehicle/weights/best.pt", recursive=True)
    if candidates:
        BEST_WEIGHTS_PATH = candidates[0]

print("Bobot terbaik tersimpan di:", BEST_WEIGHTS_PATH)
assert os.path.exists(BEST_WEIGHTS_PATH), "best.pt tidak ditemukan, cek proses training di atas."


Bobot terbaik tersimpan di: /kaggle/working/runs/detect/vehicle_counting_runs/yolov8n_vehicle/weights/best.pt


In [11]:
# (Opsional) Evaluasi cepat model pada data validasi
trained_model = YOLO(BEST_WEIGHTS_PATH)
metrics = trained_model.val(data=data_yaml_path)
print(metrics)


Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2750.1±1279.0 MB/s, size: 200.4 KB)
val: Scanning /kaggle/working/dataset_curl/valid/labels.cache... 247 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 247/247 129.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 5.6it/s 2.9s
                   all        247       1077      0.852      0.852      0.895      0.674
                   bus         17         23      0.967      0.783      0.866      0.711
                   car        173        438      0.812       0.89      0.903      0.718
             motorbike        161        550      0.798      0.811      0.873      0.523
                 truck         49         66      0.831      0.924      0.936      0.744
Speed: 1.5ms preprocess, 3.0ms in

## 5. Upload Video Input untuk Pengujian

Upload video CCTV/traffic-cam yang ingin dihitung arus kendaraannya.

- **Google Colab**: gunakan `google.colab.files.upload()` (sel pertama di bawah).
- **Kaggle**: upload video sebagai **Input dataset** (tombol "+ Add Input" di panel kanan)
  **sebelum** menjalankan Save Version — sel kedua di bawah akan **otomatis mencari** file
  video di `/kaggle/input/` sehingga tidak perlu edit path manual.


In [12]:
# PERBAIKAN PENTING: sebelumnya deteksi environment hanya pakai try/except ImportError,
# tapi ternyata modul `google.colab` bisa saja ter-import di Kaggle (ada stub-nya), sehingga
# `files.upload()` tetap terpanggil dan mengirim pesan interaktif ('colab_request') yang
# menunggu respons browser Colab. Di Kaggle (non-interaktif via Save Version), respons itu
# TIDAK PERNAH datang -> kernel hang selamanya (ini penyebab notebook "diam" berjam-jam
# sebelumnya). Sekarang deteksi environment dilakukan lebih pasti lewat folder /kaggle/input,
# sehingga sel upload Colab benar-benar dilewati (tidak pernah dieksekusi) saat di Kaggle.

import os

IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None or os.path.exists("/kaggle/input")

if not IS_KAGGLE:
    try:
        from google.colab import files
        uploaded = files.upload()
        SOURCE_VIDEO_PATH = list(uploaded.keys())[0]
        print("Video sumber (Colab):", SOURCE_VIDEO_PATH)
    except ImportError:
        print("Bukan environment Colab — lewati sel ini, gunakan sel Kaggle di bawah.")
else:
    print("Terdeteksi environment Kaggle — sel upload Colab dilewati otomatis (mencegah hang).")


Terdeteksi environment Kaggle — sel upload Colab dilewati otomatis (mencegah hang).


In [13]:
# --- Gunakan sel ini jika di Kaggle ---
# PERBAIKAN: sebelumnya path harus diisi manual di tengah notebook, tapi "Save Version"
# menjalankan notebook otomatis dari atas ke bawah tanpa jeda untuk edit manual, sehingga
# menyebabkan NameError. Sekarang path dicari otomatis di /kaggle/input/ (lokasi file yang
# di-upload lewat tombol "+ Add Input").

import glob

if "SOURCE_VIDEO_PATH" not in dir():
    video_extensions = ("*.mp4", "*.avi", "*.mov", "*.mkv")
    found_videos = []
    for ext in video_extensions:
        found_videos.extend(glob.glob(f"/kaggle/input/**/{ext}", recursive=True))

    if found_videos:
        SOURCE_VIDEO_PATH = found_videos[0]
        print("Video ditemukan otomatis:", SOURCE_VIDEO_PATH)
        if len(found_videos) > 1:
            print("Catatan: ada lebih dari 1 video di /kaggle/input, yang lain:", found_videos[1:])
            print("Jika ingin memakai yang lain, set manual: SOURCE_VIDEO_PATH = \"path/anda.mp4\"")
    else:
        raise FileNotFoundError(
            "Tidak ada file video (.mp4/.avi/.mov/.mkv) ditemukan di /kaggle/input/. "
            "Upload video dulu lewat tombol '+ Add Input' di panel kanan sebelum menjalankan "
            "Save Version, lalu jalankan ulang."
        )

print("SOURCE_VIDEO_PATH final yang dipakai:", SOURCE_VIDEO_PATH)


Video ditemukan otomatis: /kaggle/input/datasets/feeraaaa/video-traffic/Traffic CCTV Footage at Red Light Intersection _ Daytime Highway  1 Hour Full Recording.mp4
SOURCE_VIDEO_PATH final yang dipakai: /kaggle/input/datasets/feeraaaa/video-traffic/Traffic CCTV Footage at Red Light Intersection _ Daytime Highway  1 Hour Full Recording.mp4


## 6. Tracking (ByteTrack) + Unique Vehicle Counting (Supervision)

Pipeline per-frame:
1. Deteksi objek dengan YOLOv8 (`best.pt`)
2. Update tracker **ByteTrack** agar tiap kendaraan punya `tracker_id` konsisten antar frame
3. **Counting tanpa garis**: begitu sebuah `tracker_id` muncul untuk PERTAMA KALINYA, langsung
   ditambahkan ke hitungan sesuai kelasnya (`car`/`motorbike`/`bus`/`truck`). Tidak ada lagi
   konsep "melewati garis" atau arah masuk/keluar — yang dihitung murni total kendaraan unik
   yang berhasil dideteksi & di-track di sepanjang video.
4. Anotasi frame (kotak, label kelas + ID, jejak/trace, dan overlay jumlah berjalan) lalu
   ditulis ke video output.

> ⚠️ **Trade-off penting**: pendekatan ini tidak bisa membedakan arah (masuk/keluar), dan kalau
> `tracker_id` sebuah kendaraan "putus" di tengah jalan (misal ketutup kendaraan lain lama, atau
> keluar-masuk frame), kendaraan itu berpotensi terhitung lebih dari sekali karena dianggap
> kendaraan baru saat track-nya mulai lagi. Parameter ByteTrack (`lost_track_buffer`,
> `minimum_matching_threshold`) sudah di-tuning untuk meminimalkan ini.

> ⏱️ **Catatan performa**: proses ini berjalan frame-per-frame lewat model YOLO + tracker,
> jadi bisa memakan waktu lama untuk video panjang. Progress dicetak berkala di log.


In [14]:
VIDEO_INFO = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
print(VIDEO_INFO)

# PERUBAHAN: LineZone dihapus. Counting sekarang berbasis unique tracker_id per kelas,
# bukan berbasis crossing garis. Lihat penjelasan di sel markdown di atas.

# PERBAIKAN TRACKING: parameter sebelumnya (lost_track_buffer=30 ~ 1 detik,
# minimum_matching_threshold=0.8) terlalu ketat -> kendaraan yang ketutup kendaraan lain
# lebih dari 1 detik, atau motor yang gerak cepat/kecil, gampang kehilangan tracker_id
# dan dianggap "kendaraan baru" saat muncul lagi. Ini makin penting sekarang karena tanpa
# garis, setiap tracker_id baru langsung menambah hitungan -> track yang putus = overcounting.
byte_tracker = sv.ByteTrack(
    frame_rate=VIDEO_INFO.fps,
    track_activation_threshold=0.2,      # sebelumnya 0.25 -> track lebih gampang aktif
    lost_track_buffer=60,                # sebelumnya 30 -> toleransi occlusion ~2 detik (di 30fps)
    minimum_matching_threshold=0.7,      # sebelumnya 0.8 -> matching antar frame lebih toleran
)

box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.5, text_thickness=1)
trace_annotator = sv.TraceAnnotator(thickness=1, trace_length=30)

# PERBAIKAN DETEKSI: threshold confidence dibedakan per kelas.
# Dari hasil evaluasi model (Bagian 4), motorbike punya mAP50-95 paling rendah (0.535)
# dibanding car/bus/truck (>0.7) -> motor paling sering "lolos" tidak terdeteksi.
CONFIDENCE_THRESHOLD = 0.15   # threshold longgar dipakai saat inference (agar tidak keburu dibuang)
IOU_THRESHOLD = 0.65          # dinaikkan dari 0.5 -> box motor yang berdempetan tidak saling menghapus di NMS

PER_CLASS_CONFIDENCE = {
    "bus": 0.35,
    "car": 0.35,
    "motorbike": 0.20,   # diturunkan khusus untuk motor (recall paling rendah)
    "truck": 0.35,
}

# PERBAIKAN DETEKSI OBJEK KECIL: menaikkan resolusi inference (imgsz) membantu YOLO
# mendeteksi kendaraan yang jauh dari kamera / berukuran kecil di frame.
IMGSZ_INFERENCE = 960

# --- Struktur data untuk counting yang lebih stabil ---
# 1. `seen_ids` mencegah hitungan ganda untuk tracker_id yang sama.
# 2. `track_states` menahan usia track dan hanya menghitung kalau track sudah stabil
#    selama beberapa frame. Ini membantu mengurangi overcounting saat tracker putus
#    dan muncul kembali.
MIN_TRACK_FRAMES = 5
MIN_BOX_AREA = {
    "bus": 1200,
    "car": 1400,
    "motorbike": 600,
    "truck": 2200,
}

seen_ids = {name: set() for name in trained_model.names.values()}
track_states = {}
vehicle_counts = {name: 0 for name in trained_model.names.values()}

print("Kelas model:", trained_model.names)
print("Threshold per kelas:", PER_CLASS_CONFIDENCE)
print("Ukuran inference (imgsz):", IMGSZ_INFERENCE)
print("Minimum track duration (frames):", MIN_TRACK_FRAMES)
print("Minimum box area per kelas:", MIN_BOX_AREA)

VideoInfo(width=640, height=360, fps=30.0, total_frames=127790)
Kelas model: {0: 'bus', 1: 'car', 2: 'motorbike', 3: 'truck'}
Threshold per kelas: {'bus': 0.35, 'car': 0.35, 'motorbike': 0.2, 'truck': 0.35}
Ukuran inference (imgsz): 960
Minimum track duration (frames): 5
Minimum box area per kelas: {'bus': 1200, 'car': 1400, 'motorbike': 600, 'truck': 2200}


The `ByteTrack` was deprecated since v0.28.0. It will be removed in v0.30.0.


In [15]:
def process_frame(frame: np.ndarray, index: int) -> np.ndarray:
    # 1. Deteksi objek dengan threshold yang lebih longgar
    result = trained_model(
        frame,
        conf=CONFIDENCE_THRESHOLD,
        iou=IOU_THRESHOLD,
        imgsz=IMGSZ_INFERENCE,
        verbose=False,
    )[0]
    detections = sv.Detections.from_ultralytics(result)

    # 1b. Filter ulang per kelas memakai threshold yang lebih sesuai
    if len(detections) > 0:
        class_names = [trained_model.names[c] for c in detections.class_id]
        keep_mask = np.array([
            conf >= PER_CLASS_CONFIDENCE.get(name, CONFIDENCE_THRESHOLD)
            for name, conf in zip(class_names, detections.confidence)
        ])
        detections = detections[keep_mask]

    # 2. Update tracker ByteTrack -> setiap deteksi mendapat tracker_id
    detections = byte_tracker.update_with_detections(detections)

    # 3. Counting yang lebih stabil: hanya hitung setelah track sudah ada beberapa frame
    #    dan ukuran kotak cukup besar. Ini membantu mengurangi overcounting saat tracker putus.
    if len(detections) > 0:
        for class_id, tracker_id, confidence, xyxy in zip(
            detections.class_id,
            detections.tracker_id,
            detections.confidence,
            detections.xyxy,
        ):
            if tracker_id is None:
                continue

            class_name = trained_model.names[class_id]
            x1, y1, x2, y2 = xyxy
            box_area = max(0.0, (x2 - x1) * (y2 - y1))

            if box_area < MIN_BOX_AREA.get(class_name, 0):
                continue

            track_key = (class_name, int(tracker_id))
            track_states[track_key] = track_states.get(track_key, 0) + 1

            if track_states[track_key] >= MIN_TRACK_FRAMES and int(tracker_id) not in seen_ids[class_name]:
                seen_ids[class_name].add(int(tracker_id))
                vehicle_counts[class_name] += 1

    # 4. Buat label: "<class> #<tracker_id> <conf>"
    labels = [
        f"{trained_model.names[class_id]} #{tracker_id} {confidence:0.2f}"
        for class_id, tracker_id, confidence in zip(
            detections.class_id, detections.tracker_id, detections.confidence
        )
    ]

    annotated_frame = frame.copy()
    annotated_frame = trace_annotator.annotate(annotated_frame, detections=detections)
    annotated_frame = box_annotator.annotate(annotated_frame, detections=detections)
    annotated_frame = label_annotator.annotate(annotated_frame, detections=detections, labels=labels)

    # 5. Overlay jumlah berjalan (running total) di pojok kiri atas
    y_offset = 30
    total_so_far = sum(vehicle_counts.values())
    cv2.putText(
        annotated_frame,
        f"Total: {total_so_far}",
        (10, y_offset),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2,
        cv2.LINE_AA,
    )
    for name, count in vehicle_counts.items():
        y_offset += 25
        cv2.putText(
            annotated_frame,
            f"{name}: {count}",
            (10, y_offset),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )

    return annotated_frame


In [16]:
# PERBAIKAN PALING PENTING: sebelumnya TEST_DURATION_SECONDS = 20 masih aktif saat
# dianggap sebagai hasil final. Akibatnya cuma 600 frame dari total 127.790 frame video
# sumber (~0.47%!) yang benar-benar diproses -> hasil counting jadi jauh lebih kecil dari
# kondisi asli. Sekarang di-set None supaya seluruh video diproses.
#
# Catatan: proses full video akan memakan waktu jauh lebih lama dibanding uji coba 20 detik.
# Kalau cuma mau tes cepat pipeline-nya jalan, boleh set balik ke angka kecil (mis. 20)
# SEMENTARA, tapi pastikan diubah ke None lagi sebelum dianggap hasil final.
TARGET_VIDEO_PATH = "vehicle_counting_result.mp4"

TEST_DURATION_SECONDS = None
MAX_FRAMES = int(TEST_DURATION_SECONDS * VIDEO_INFO.fps) if TEST_DURATION_SECONDS else None

total_frames = VIDEO_INFO.total_frames
frames_to_process = min(MAX_FRAMES, total_frames) if MAX_FRAMES else total_frames
print(f"Total frame video: {total_frames} | Akan diproses: {frames_to_process} frame "
      f"(~{frames_to_process / VIDEO_INFO.fps / 60:.1f} menit)")

frame_generator = sv.get_video_frames_generator(source_path=SOURCE_VIDEO_PATH)

with sv.VideoSink(target_path=TARGET_VIDEO_PATH, video_info=VIDEO_INFO) as sink:
    for i, frame in enumerate(frame_generator):
        if MAX_FRAMES and i >= MAX_FRAMES:
            print(f"Berhenti di frame {i} sesuai batas uji coba (TEST_DURATION_SECONDS={TEST_DURATION_SECONDS}).")
            break

        annotated_frame = process_frame(frame, i)
        sink.write_frame(annotated_frame)

        # Progress print setiap 150 frame (~5 detik video) atau di frame terakhir.
        if i % 150 == 0 or i == frames_to_process - 1:
            pct = (i + 1) / frames_to_process * 100
            print(f"Progress: frame {i + 1}/{frames_to_process} ({pct:.1f}%) | "
                  f"Total kendaraan unik sejauh ini: {sum(vehicle_counts.values())}", flush=True)

print("\nSelesai. Video hasil deteksi & counting tersimpan di:", TARGET_VIDEO_PATH)
print("\n=== REKAP AKHIR: JUMLAH KENDARAAN UNIK PER JENIS ===")
for name, count in vehicle_counts.items():
    print(f"  {name}: {count}")
print(f"  TOTAL SEMUA JENIS: {sum(vehicle_counts.values())}")


Total frame video: 127790 | Akan diproses: 127790 frame (~71.0 menit)
Progress: frame 1/127790 (0.0%) | Total kendaraan unik sejauh ini: 0
Progress: frame 151/127790 (0.1%) | Total kendaraan unik sejauh ini: 28
Progress: frame 301/127790 (0.2%) | Total kendaraan unik sejauh ini: 45
Progress: frame 451/127790 (0.4%) | Total kendaraan unik sejauh ini: 54
Progress: frame 601/127790 (0.5%) | Total kendaraan unik sejauh ini: 75
Progress: frame 751/127790 (0.6%) | Total kendaraan unik sejauh ini: 86
Progress: frame 901/127790 (0.7%) | Total kendaraan unik sejauh ini: 99
Progress: frame 1051/127790 (0.8%) | Total kendaraan unik sejauh ini: 106
Progress: frame 1201/127790 (0.9%) | Total kendaraan unik sejauh ini: 122
Progress: frame 1351/127790 (1.1%) | Total kendaraan unik sejauh ini: 138
Progress: frame 1501/127790 (1.2%) | Total kendaraan unik sejauh ini: 151
Progress: frame 1651/127790 (1.3%) | Total kendaraan unik sejauh ini: 158
Progress: frame 1801/127790 (1.4%) | Total kendaraan unik s

## 7. Rekap Hitungan per Kelas Kendaraan

Karena counting sekarang berbasis **unique tracker_id** (bukan crossing garis), rekap per
kelas sudah otomatis tersimpan di dictionary `vehicle_counts` selama proses berjalan
(diperbarui langsung di dalam `process_frame`). Sel di bawah hanya menampilkan ulang hasil
akhirnya dalam bentuk tabel ringkas.


In [17]:
print("=== JUMLAH KENDARAAN UNIK TERDETEKSI PER JENIS (seluruh video) ===\n")

total = sum(vehicle_counts.values())
for name, count in vehicle_counts.items():
    pct = (count / total * 100) if total > 0 else 0
    print(f"  {name:<12}: {count:>5}  ({pct:5.1f}%)")

print(f"\n  {'TOTAL':<12}: {total:>5}")

# (Opsional) Tampilkan dalam bentuk tabel pandas jika tersedia
try:
    import pandas as pd
    df_counts = pd.DataFrame(
        [(name, count) for name, count in vehicle_counts.items()],
        columns=["Jenis Kendaraan", "Jumlah Unik"]
    ).sort_values("Jumlah Unik", ascending=False).reset_index(drop=True)
    display(df_counts)
except ImportError:
    pass


=== JUMLAH KENDARAAN UNIK TERDETEKSI PER JENIS (seluruh video) ===

  bus         :     3  (  0.1%)
  car         :   535  (  9.3%)
  motorbike   :  5106  ( 89.0%)
  truck       :    92  (  1.6%)

  TOTAL       :  5736


,Jenis Kendaraan,Jumlah Unik
0,motorbike,5106
1,car,535
2,truck,92
3,bus,3


## 8. Unduh Output Akhir

Output akhir notebook ini:
- `best.pt` — bobot model YOLOv8 hasil fine-tuning
- `vehicle_counting_result.mp4` — video hasil deteksi, lengkap dengan tracker ID dan counter

- **Google Colab**: gunakan `files.download()` (sel di bawah).
- **Kaggle**: kedua file otomatis muncul di panel **Output** (kanan) setelah **Save Version**
  selesai — tinggal klik file tersebut untuk mengunduhnya, tidak perlu `files.download()`.


In [18]:
import os

IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None or os.path.exists("/kaggle/input")

if not IS_KAGGLE:
    try:
        from google.colab import files
        files.download(BEST_WEIGHTS_PATH)
        files.download(TARGET_VIDEO_PATH)
    except ImportError:
        print("Bukan environment Colab — file otomatis tersedia di panel Output Kaggle setelah Save Version.")
        print("Lokasi file:", BEST_WEIGHTS_PATH, "&", TARGET_VIDEO_PATH)
else:
    print("Terdeteksi environment Kaggle — file otomatis tersedia di panel Output setelah Save Version.")
    print("Lokasi file:", BEST_WEIGHTS_PATH, "&", TARGET_VIDEO_PATH)


Terdeteksi environment Kaggle — file otomatis tersedia di panel Output setelah Save Version.
Lokasi file: /kaggle/working/runs/detect/vehicle_counting_runs/yolov8n_vehicle/weights/best.pt & vehicle_counting_result.mp4


---
### Ringkasan Perbaikan pada Versi Ini
1. **`NameError: name 'DATASET_LOCATION' is not defined`** — diperbaiki dengan mengaktifkan
   `DATASET_LOCATION = "dataset_curl"` secara default (sebelumnya ter-*comment*).
2. **`NameError: name 'rf' is not defined`** — sel diagnostik SDK Roboflow sekarang bersifat
   *self-contained* (membuat `rf` sendiri di dalam sel yang sama).
3. **`BadZipFile: File is not a zip file`** (Opsi A) — diberi try/except dengan pesan diagnostik
   yang jelas (workspace/project/versi mana yang gagal), dan Opsi B (`curl`) dijadikan jalur
   utama karena sudah terbukti berhasil di lingkungan kamu.
4. Ditambahkan dukungan **lintas platform** (Colab & Kaggle) untuk upload video dan download hasil,
   karena kamu menjalankan notebook ini di Kaggle, bukan Colab.
5. Ditambahkan sel opsional perbaikan path `data.yaml` (Bagian 3) untuk berjaga-jaga jika training
   gagal menemukan gambar akibat path relatif yang salah.
6. **`AssertionError: best.pt tidak ditemukan`** — path `BEST_WEIGHTS_PATH` sebelumnya ditulis
   manual (`vehicle_counting_runs/yolov8n_vehicle/weights/best.pt`) dan tidak selalu cocok dengan
   lokasi asli hasil training (Ultralytics kadang menambahkan prefix `runs/detect/...`). Sekarang
   path diambil langsung dari `model.trainer.save_dir` (selalu akurat), dengan pencarian otomatis
   (`glob`) sebagai jaring pengaman jika tetap tidak ketemu.
7. **`NameError: name 'SOURCE_VIDEO_PATH' is not defined'`** (potensial di Kaggle) — sebelumnya
   sel Kaggle hanya print pengingat tanpa benar-benar mengisi variabel, sehingga akan gagal saat
   dijalankan otomatis lewat Save Version. Sekarang path video dicari otomatis di `/kaggle/input/`
   dan langsung memberi pesan error yang jelas (`FileNotFoundError`) jika belum ada video di-upload,
   alih-alih `NameError` yang membingungkan di sel berikutnya.
8. **Notebook terlihat "diam/macet" berjam-jam persis setelah evaluasi model** — **ini akar
   masalah sesungguhnya**, bukan soal video panjang seperti dugaan awal. Sel upload video Colab
   sebelumnya hanya mengandalkan `try/except ImportError` untuk deteksi environment, tapi modul
   `google.colab` ternyata bisa ter-*import* di Kaggle juga. Akibatnya `files.upload()` tetap
   terpanggil dan mengirim pesan interaktif (`colab_request`) yang menunggu respons browser Colab
   — respons yang **tidak akan pernah datang** di Kaggle non-interaktif (Save Version), sehingga
   kernel hang selamanya. Sekarang deteksi environment memakai `os.path.exists("/kaggle/input")`,
   sehingga sel Colab **benar-benar tidak pernah dieksekusi** saat berjalan di Kaggle.
9. Video processing (Bagian 6) juga tetap diperbaiki dengan progress print + batas durasi uji
   coba (`TEST_DURATION_SECONDS`), sebagai lapisan kedua supaya proses panjang tetap terlihat
   progresnya secara real-time di Kaggle Logs.

### Catatan Tambahan
- **Kalibrasi garis hitung**: sesuaikan `LINE_START` / `LINE_END` dengan posisi jalan pada CCTV/kamera sesungguhnya.
- **Kelas kendaraan**: cek isi `CLASS_NAMES` di Bagian 3 — kalau namanya bahasa Inggris (`car`, `motorcycle`, `truck`, `bus`), label & rekap counting akan otomatis mengikuti nama itu apa adanya.
- **Multi-line counting**: untuk simpang/jalur banyak arah, buat beberapa objek `sv.LineZone` (satu per garis/arah) dan panggil `.trigger()` untuk masing-masing di dalam `process_frame`.
- **Keamanan API key**: sebelum notebook ini dibagikan ke tim lain atau diunggah ke repo, cabut/rotasi API key Roboflow yang tertulis eksplisit di sel opsional.


---
### 🔧 Ringkasan Perbaikan Akurasi Deteksi & Counting (Round 2)

Masalah utama sebelumnya: video hasil yang dievaluasi cuma 20 detik dari total video
sumber ~71 menit (**0,47% saja**), sehingga terlihat seperti "banyak kendaraan tidak
kehitung" padahal sebagian besar video belum pernah diproses. Perbaikan berikut mengatasi
itu dan beberapa penyebab lain di balik deteksi yang meleset:

1. **`TEST_DURATION_SECONDS = None`** (Bagian 6, sel proses utama) — sekarang memproses
   seluruh video, bukan cuma 20 detik pertama. Ini perbaikan dengan dampak terbesar.
2. **Tuning ByteTrack** — `lost_track_buffer` dinaikkan 30 → 60 (toleransi kendaraan
   ketutup objek lain lebih lama), `minimum_matching_threshold` diturunkan 0.8 → 0.7
   (matching antar frame lebih toleran untuk kendaraan yang gerak cepat/kecil seperti motor).
3. **Threshold confidence per kelas** (`PER_CLASS_CONFIDENCE`) — motorbike diturunkan ke
   0.20 karena mAP50-95 motor paling rendah di antara 4 kelas (0.535 vs >0.7 untuk
   car/bus/truck), sehingga motor yang sebelumnya "hampir terdeteksi" tapi terbuang oleh
   threshold seragam 0.3 sekarang bisa ikut kehitung. Kelas lain tetap di 0.35 supaya
   tidak menambah false positive.
4. **`IMGSZ_INFERENCE = 960`** — resolusi saat inference dinaikkan dari resolusi asli video
   (640) agar kendaraan yang jauh dari kamera (kecil di frame) lebih mudah terdeteksi.
   Bisa dinaikkan lagi ke 1280 kalau waktu proses masih memungkinkan.
5. Interval progress print diperbesar (30 → 150 frame) karena sekarang memproses full
   video yang jauh lebih panjang — supaya log tidak kebanjiran print.

**Catatan:** perbaikan ini semua di sisi kode/parameter, belum menyentuh dataset atau
retraining model. Kalau setelah ini motor di kondisi padat/berdempetan masih banyak yang
lolos, langkah berikutnya adalah menambah data training motor (khususnya kondisi ramai
mirip video ini) atau upgrade ke `yolov8s.pt`.


VIDEO_INFO = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
print(VIDEO_INFO)

# PERUBAHAN: LineZone dihapus. Counting sekarang berbasis unique tracker_id per kelas,
# bukan berbasis crossing garis. Lihat penjelasan di sel markdown di atas.

# PERBAIKAN TRACKING: parameter sebelumnya (lost_track_buffer=30 ~ 1 detik,
# minimum_matching_threshold=0.8) terlalu ketat -> kendaraan yang ketutup kendaraan lain
# lebih dari 1 detik, atau motor yang gerak cepat/kecil, gampang kehilangan tracker_id
# dan dianggap "kendaraan baru" saat muncul lagi. Ini makin penting sekarang karena tanpa
# garis, setiap tracker_id baru langsung menambah hitungan -> track yang putus = overcounting.
byte_tracker = sv.ByteTrack(
    frame_rate=VIDEO_INFO.fps,
    track_activation_threshold=0.2,      # sebelumnya 0.25 -> track lebih gampang aktif
    lost_track_buffer=60,                # sebelumnya 30 -> toleransi occlusion ~2 detik (di 30fps)
    minimum_matching_threshold=0.7,      # sebelumnya 0.8 -> matching antar frame lebih toleran
)

box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.5, text_thickness=1)
trace_annotator = sv.TraceAnnotator(thickness=1, trace_length=30)

# PERBAIKAN DETEKSI: threshold confidence dibedakan per kelas.
# Dari hasil evaluasi model (Bagian 4), motorbike punya mAP50-95 paling rendah (0.535)
# dibanding car/bus/truck (>0.7) -> motor paling sering "lolos" tidak terdeteksi.
CONFIDENCE_THRESHOLD = 0.15   # threshold longgar dipakai saat inference (agar tidak keburu dibuang)
IOU_THRESHOLD = 0.65          # dinaikkan dari 0.5 -> box motor yang berdempetan tidak saling menghapus di NMS

PER_CLASS_CONFIDENCE = {
    "bus": 0.35,
    "car": 0.35,
    "motorbike": 0.20,   # diturunkan khusus untuk motor (recall paling rendah)
    "truck": 0.35,
}

# PERBAIKAN DETEKSI OBJEK KECIL: menaikkan resolusi inference (imgsz) membantu YOLO
# mendeteksi kendaraan yang jauh dari kamera / berukuran kecil di frame.
IMGSZ_INFERENCE = 960

# --- Struktur data untuk counting yang lebih stabil ---
# 1. `seen_ids` mencegah hitungan ganda untuk tracker_id yang sama.
# 2. `track_states` menahan usia track dan hanya menghitung kalau track sudah stabil
#    selama beberapa frame. Ini membantu mengurangi overcounting saat tracker putus
#    dan muncul kembali.
MIN_TRACK_FRAMES = 5
MIN_BOX_AREA = {
    "bus": 1200,
    "car": 1400,
    "motorbike": 600,
    "truck": 2200,
}

seen_ids = {name: set() for name in trained_model.names.values()}
track_states = {}
vehicle_counts = {name: 0 for name in trained_model.names.values()}

print("Kelas model:", trained_model.names)
print("Threshold per kelas:", PER_CLASS_CONFIDENCE)
print("Ukuran inference (imgsz):", IMGSZ_INFERENCE)
print("Minimum track duration (frames):", MIN_TRACK_FRAMES)
print("Minimum box area per kelas:", MIN_BOX_AREA)